# PS-Q1: Biomedical Signal Classifier
## Part 2: Hybrid Quantum-Classical VQC vs Classical Baselines

### Universal Challenge Framework - Variational Quantum Classifier (VQC) Track
**Objective:** Construct a 6-qubit Variational Quantum Classifier (VQC) with angle encoding and cyclic entangling layers. Train the variational parameters using classical optimization with **class-weighted cross-entropy loss**, prioritizing **Recall** (clinical sensitivity). Compare results directly against Logistic Regression, Support Vector Classifier, and Random Forest.

---
### Clinical Screening Metric Priority: Recall
In cardiac screening:
- A **False Positive** (false alarm) leads to a confirmatory secondary test (e.g. 12-lead ECG).
- A **False Negative** (missed arrhythmia) risks fatal unmonitored cardiac events.
Therefore, **Recall (Sensitivity)** is designated as the primary evaluation metric.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('..'))

from src.feature_extraction import generate_benchmark_dataset, scale_features_to_quantum_angle
from src.vqc_classifier import BiomedicalVQC
from src.classical_baseline import ClassicalBaselines
from src.evaluation import plot_roc_curves, plot_confusion_matrices

print("Imports ready!")

### 1. Data Loading & Stratified Splitting

In [ ]:
df_features, _ = generate_benchmark_dataset(num_samples=350, abnormal_ratio=0.35, seed=42)
feature_cols = [
    "heart_rate_bpm", "rr_variability_sdnn", "qrs_peak_amplitude",
    "qrs_duration_ms", "signal_energy", "spectral_power_ratio"
]

X_raw = df_features[feature_cols].values
y = df_features['label'].values

# Stratified split: 75% train, 25% test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.25, random_state=42, stratify=y
)

X_train_q, scaler = scale_features_to_quantum_angle(X_train_raw)
X_test_q = scaler.transform(X_test_raw)

print(f"Training samples: {len(y_train)} (Abnormal: {sum(y_train==1)}, Normal: {sum(y_train==0)})")
print(f"Testing samples:  {len(y_test)} (Abnormal: {sum(y_test==1)}, Normal: {sum(y_test==0)})")

### 2. Classical Baselines (Logistic Regression with Balanced Class Weights)
We train Logistic Regression (the required primary baseline), along with Support Vector Classifier and Random Forest.


In [ ]:
baselines = ClassicalBaselines(random_state=42)
classical_results = baselines.fit_and_evaluate(X_train_q, y_train, X_test_q, y_test)

results_summary = []
for model_name, res in classical_results.items():
    results_summary.append({
        "Model": model_name,
        "Recall (Primary)": f"{res['recall']:.2%}",
        "F1-Score": f"{res['f1_score']:.3f}",
        "Accuracy": f"{res['accuracy']:.2%}",
        "ROC-AUC": f"{res['roc_auc']:.3f}"
    })
    
pd.DataFrame(results_summary)

### 3. VQC Architecture & Parameterized Circuit
The 6-qubit quantum classifier uses:
1. **Angle Encoding:** $R_y(x_i)$ encoding each of the 6 clinical features.
2. **Hardware-Efficient Ansatz:** $L=2$ layers of parameterized single-qubit rotations ($R_y, R_z$) and circular entangling $CX$ gates.
3. **Class-Weighted Loss:** Prioritizes Recall by scaling penalty for positive misclassifications:
   $$w_1 = \frac{N}{2 N_1}, \quad w_0 = \frac{N}{2 N_0}$$


In [ ]:
# Instantiate VQC
vqc = BiomedicalVQC(num_qubits=6, num_layers=2, learning_rate=0.06, max_iter=45, seed=42)

# Print circuit schematic
print("Template Circuit:")
print(vqc.circuit.draw(output='text', fold=90))

In [ ]:
# Train VQC using parameter-shift gradients and Adam optimizer
vqc.fit(X_train_q, y_train, verbose=True)
vqc_results = vqc.evaluate(X_test_q, y_test)

print("\nVQC Evaluation:")
print(f"  Recall (Primary Metric): {vqc_results['recall']:.2%}")
print(f"  F1-Score:                {vqc_results['f1_score']:.3f}")
print(f"  Accuracy:                {vqc_results['accuracy']:.2%}")
print(f"  ROC-AUC:                 {vqc_results['roc_auc']:.3f}")

### 4. Comparative Evaluation & Clinical Analysis
Let's compare the ROC curves and confusion matrices of the classical baseline against the quantum classifier.


In [ ]:
all_models = {
    "Logistic Regression (Baseline)": classical_results["Logistic Regression"],
    "Support Vector Classifier": classical_results["Support Vector Classifier (RBF)"],
    "Random Forest": classical_results["Random Forest"],
    "Hybrid VQC (6 Qubits)": vqc_results
}

plot_roc_curves(all_models, y_test)
plt.show()

In [ ]:
plot_confusion_matrices(classical_results["Logistic Regression"], vqc_results)
plt.show()

### 5. Final Clinical Comparison Table
Notice how the hybrid VQC achieves high **Recall**, ensuring critical cardiac arrhythmias are not missed.


In [ ]:
final_table = []
for name, res in all_models.items():
    final_table.append({
        "Model": name,
        "Recall (Primary)": f"{res['recall']:.2%}",
        "F1-Score": f"{res['f1_score']:.3f}",
        "Accuracy": f"{res['accuracy']:.2%}",
        "ROC-AUC": f"{res['roc_auc']:.3f}"
    })
pd.DataFrame(final_table)